# Sheep annotator comparison: IgBLAST vs abstar (200 reads)

Сравнение уже готовых smoke-аннотаций для овцы `PRJNA900592`, sample `SRR22279249`, первые/отобранные 200 merged sequences.

Важно: сейчас оба аннотатора запускались без sheep-specific germline DB:
- IgBLAST — через mouse germline DB
- abstar — через `c57bl6`

Поэтому V/D/J gene-call agreement интерпретировать осторожно; главный переносимый показатель для сравнения — `cdr3_aa`.


In [ ]:
# CELL 1: Setup
import csv, json
from pathlib import Path
from collections import Counter

DATASET = "PRJNA900592"
SAMPLE = "SRR22279249"
N = 200
BASE = Path("/data/user/epishkin/results") / DATASET
COMPARE = BASE / "annotator_compare"

IG_TSV = COMPARE / "output" / "igblast" / f"{SAMPLE}_{N}_igblast.tsv"
AB_TSV = COMPARE / "output" / "abstar" / SAMPLE / f"{SAMPLE}_{N}.tsv"
REPORT = COMPARE / "comparison_report_sheep.json"

print("DATASET:", DATASET)
print("SAMPLE:", SAMPLE)
print("IgBLAST:", IG_TSV, "exists=", IG_TSV.exists(), "size=", IG_TSV.stat().st_size if IG_TSV.exists() else None)
print("abstar: ", AB_TSV, "exists=", AB_TSV.exists(), "size=", AB_TSV.stat().st_size if AB_TSV.exists() else None)


In [ ]:
# CELL 2: If needed, copy existing smoke outputs into annotator_compare layout
# Existing outputs were produced earlier under igblast_smoke/ and abstar_smoke/.
# This cell only copies them into the canonical compare directory; it does not rerun annotation.

src_ig = BASE / "igblast_smoke" / "output" / f"{SAMPLE}_{N}_igblast.tsv"
src_ab = BASE / "abstar_smoke" / "output" / SAMPLE / "airr" / f"{SAMPLE}_{N}.tsv"
src_fa = BASE / "igblast_smoke" / "input" / f"{SAMPLE}_{N}.fasta"

(COMPARE / "input").mkdir(parents=True, exist_ok=True)
(IG_TSV.parent).mkdir(parents=True, exist_ok=True)
(AB_TSV.parent).mkdir(parents=True, exist_ok=True)

for src, dst in [(src_fa, COMPARE / "input" / src_fa.name), (src_ig, IG_TSV), (src_ab, AB_TSV)]:
    if dst.exists() and dst.stat().st_size > 0:
        print("SKIP exists:", dst)
    elif src.exists() and src.stat().st_size > 0:
        dst.write_bytes(src.read_bytes())
        print("COPIED:", src, "->", dst)
    else:
        print("MISSING SOURCE:", src)

print("
Final files:")
for p in [IG_TSV, AB_TSV]:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)


In [ ]:
# CELL 3: Load TSVs and check shared sequence IDs

def load_tsv(path):
    rows = {}
    with open(path, newline="") as f:
        reader = csv.DictReader(f, delimiter="	")
        for row in reader:
            rows[row["sequence_id"]] = row
    return rows

if not IG_TSV.exists():
    raise FileNotFoundError(f"IgBLAST TSV missing: {IG_TSV}")
if not AB_TSV.exists():
    raise FileNotFoundError(f"abstar TSV missing: {AB_TSV}")

igrows = load_tsv(IG_TSV)
abrows = load_tsv(AB_TSV)
shared = sorted(set(igrows) & set(abrows))

print(f"IgBLAST reads: {len(igrows)}")
print(f"abstar reads:  {len(abrows)}")
print(f"Shared reads:  {len(shared)}")
print("First 5 shared IDs:", shared[:5])


In [ ]:
# CELL 4: Compare productivity, V/D/J calls, and CDR3 AA

def is_productive(row):
    return row.get("productive", "").strip().lower() in {"t", "true", "1", "yes"}

def pct(a, b):
    return round((a / b * 100), 1) if b else 0.0

ig_prod = sum(is_productive(r) for r in igrows.values())
ab_prod = sum(is_productive(r) for r in abrows.values())

stats = {
    "dataset": DATASET,
    "sample": SAMPLE,
    "reads_requested": N,
    "igblast": {"total": len(igrows), "productive": ig_prod, "productive_pct": pct(ig_prod, len(igrows))},
    "abstar": {"total": len(abrows), "productive": ab_prod, "productive_pct": pct(ab_prod, len(abrows)), "germline_db": "c57bl6"},
    "shared_reads": len(shared),
}

for field in ["v_call", "d_call", "j_call", "cdr3_aa"]:
    agree = sum(igrows[s].get(field, "").strip() == abrows[s].get(field, "").strip() for s in shared)
    stats[field + "_agree"] = agree
    stats[field + "_agree_pct"] = pct(agree, len(shared))

print("Productive IgBLAST:", f"{ig_prod}/{len(igrows)}", f"({stats['igblast']['productive_pct']}%)")
print("Productive abstar: ", f"{ab_prod}/{len(abrows)}", f"({stats['abstar']['productive_pct']}%)")
print()
print(f"On {len(shared)} shared reads:")
for field in ["v_call", "d_call", "j_call", "cdr3_aa"]:
    print(f"  {field:8s}: {stats[field + '_agree']}/{len(shared)} ({stats[field + '_agree_pct']}%)")


In [ ]:
# CELL 5: Show call distributions and first examples

def top_counts(rows, field, n=10):
    c = Counter((r.get(field, "") or "<EMPTY>").strip() or "<EMPTY>" for r in rows.values())
    return c.most_common(n)

print("Top IgBLAST V calls:")
for k, v in top_counts(igrows, "v_call"):
    print(f"  {v:3d}  {k}")

print("
Top abstar V calls:")
for k, v in top_counts(abrows, "v_call"):
    print(f"  {v:3d}  {k}")

print("
First 10 shared reads: IgBLAST -> abstar")
for sid in shared[:10]:
    print(f"{sid}")
    print("  V:", igrows[sid].get("v_call", ""), "->", abrows[sid].get("v_call", ""))
    print("  J:", igrows[sid].get("j_call", ""), "->", abrows[sid].get("j_call", ""))
    print("  CDR3_AA:", igrows[sid].get("cdr3_aa", ""), "->", abrows[sid].get("cdr3_aa", ""))


In [ ]:
# CELL 6: Save JSON report
report = {
    **stats,
    "notes": [
        "Sheep-specific germline DB is not available in this task.",
        "IgBLAST output was produced using mouse germline DB.",
        "abstar output was produced using c57bl6 germline DB.",
        "Interpret V/D/J call agreement cautiously; CDR3_AA is the most portable comparison field."
    ],
}
REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("WROTE:", REPORT)
print(json.dumps(report, indent=2, ensure_ascii=False))
